In [2]:
# Search_All_Jewish_Cemeteries.py
# 5/10/2025

from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from zenrows import ZenRowsClient
import requests
import math
import numpy as np
import urllib3
import inspect
# from IPython.display import display


# def maximize_screen():
#     from IPython.display import display, HTML
#     display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def left(s, amount):
    return s[:amount]
def right(s, amount):
    return s[-amount:]
def mid(s, offset, amount):
    return s[offset:offset+amount]

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))

def timestamp():
    today=str(datetime.date.today()).split('-')
    year=today[0]
    month=today[1]
    day=today[2]
    a=str(datetime.datetime.now())
    time_split=re.split(':| ',a)
    hour=time_split[1]
    timestamp=year+month+day+hour
    return timestamp

def convert_date_z(date): 
    date_t= pd.to_datetime(date, utc=True)
    date_out=date_t.dt.strftime("%m/%d/%Y")
    return date_out

def convert_datetime_t(event_date_t):
    no_time='2000-01-01T00:00:00'
    no_time_dt=dt.strptime(no_time, "%Y-%m-%dT%H:%M:%S") 
    event_date_t = event_date_t.fillna(no_time)
    event_date_dt=pd.to_datetime(event_date_t, format="%Y-%m-%dT%H:%M:%S")
    event_date_index=event_date_dt.dt.strftime("%Y%m%d")
    event_date =event_date_dt.dt.strftime("%m/%d/%Y")
    event_date_rnd_dt = event_date_dt + timedelta(minutes=30)
    event_time_index=event_date_rnd_dt.dt.strftime('%H')  
    event_time = event_date_dt.dt.strftime('%H:%M')
    return     event_date,event_date_index,event_time,event_time_index 



def random_sleep(call_count): 
    global sleep_time0,listings_df
    if call_count%100==0:
        sleep_time =sleep_time0
        now_date = datetime.datetime.now() + timedelta(seconds=sleep_time)
        sleep_end_date=now_date.strftime("%m/%d/%Y %H:%M:%S")    
        print(f"\t\t\tSleeping for {sleep_time:.0f} seconds • Sleeping till: {sleep_end_date}")
        # listings_df.to_csv(fileOut,mode="w", index=False,header=True)
        # listings_df.to_csv(fileOut0,mode="w", index=False,header=True) 
    else:
        sleep_time = random.uniform(3, 8)
    return sleep_time 

def get_total_cemeteries(url):
    global my_header
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    total_cemeteries=capture_json['data']['cemeteries']['total']
    return total_cemeteries

def get_grave_count(url):
    global status_code
    capture_json=[]
    grave_count=0
    response = requests.request("GET", url, headers=my_header) 
    status_code=response.status_code
    url_source = response.text 
    if status_code!=200:print('\tstatus_code:',status_code, 'url',url);return capture_json,grave_count
    beautifulSoupText = BeautifulSoup(url_source, 'html.parser') 
    specific_tag_text = beautifulSoupText.get_text()
    capture_json=json.loads(specific_tag_text) 
    grave_count=capture_json['total']
    return capture_json,grave_count
    
def extract_grave_data(capture_json):
    global field_list_set,field_list
    collection_df=pd.json_normalize(capture_json['collection'])
    collection_columns_set=set(list(collection_df))
    missing_columns=field_list_set-collection_columns_set
    collection_df[list(missing_columns)]=np.nan
    grave_frame_df=collection_df[field_list]
    return grave_frame_df

def clean_dataframe(graves_df):
    global field_list
    for column in field_list:
        graves_df[column]= graves_df[column].astype(str).replace(r"\,", ' * ', regex=True).str.strip()
        graves_df[column]= graves_df[column].astype(str).replace(r"\'", '', regex=True).str.strip()
    delete_list=[r'\"',r"\'",r'\?',r'\[',r'\]']
    for item in delete_list:
        for column in field_list:
            graves_df[column]= graves_df[column].astype(str).replace(item, ' ', regex=True).str.strip()
    return graves_df

def time_lapse(tic):
    toc=datetime.datetime.now()
    processing_time=toc-tic1
    processing_time_seconds=int(processing_time.total_seconds())
    return processing_time_seconds

#*************** Append  *******************************
append=0
# append_flag=input('To Append Press 1')
# if append_flag=='1':append=1
#*************** Append  *******************************

my_header = {
  'priority': 'u=1, i',
  'sec-ch-ua': '"Google Chrome";v="135", "Not-A.Brand";v="8", "Chromium";v="135"',
  'sec-ch-ua-platform': '"Windows"',
  'sec-fetch-mode': 'cors',
  'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/135.0.0.0 Safari/537.36',
  'Cookie': '__cf_bm=O55WB_mmMt72i8KoQuzWliizNTb6J7zg2RVHXqbn.vc-1745620049-1.0.1.1-IDSlPNW4kRjBMAr5n8M2obOoGp9GAod_BHbB997Ze2F0YUm2DvAFh5TaijsQhPWR0iyIyFoaSy.p1YDke07ZttHHC83SJpQCEBFmaeImKIc; _cfuvid=zhvP.IczaAr9sd4YVjWjTqBayGxsei_Ntl0ANgSCCBg-1745620049033-0.0.1.1-604800000; ipLoc=us; preferredLanguage=en'
}

maximize_screen()
timestamp=timestamp()
collection_date0=datetime.datetime.now().strftime("%m/%d/%Y")
collection_date_dt=pd.to_datetime(collection_date0)
collection_date=collection_date0

start_time=datetime.datetime.now()
event_time=start_time

process_name='Search_All_Jewish_Cemeteries '
print('**Starting *** ',process_name,' *** , Time:',start_time,' **\n')
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileInput="FAG_Cemetrety_List"
fileOutput="FAG_Cemetrety_Cohen_Counts"

if append==1:fileIn='FAG_Cemetrety_List_X'


fileIn=DirMain+"OutputData\\Cemeteries\\"+fileInput+".csv"
fileOut=DirMain+"OutputData\\Cemeteries\\"+fileOutput+"_"+timestamp+".csv"

fileOut0=DirMain+"OutputData\\Cemeteries\\"+fileOutput+".csv"
fileOutInt=DirMain+"OutputData\\Cemeteries\\"+fileOutput+"_Interim.csv"

cemetery_info_df=pd.read_csv(fileIn)
cemetery_id_list=cemetery_info_df["Cemetery_ID"]
cemetery_name_list=cemetery_info_df["Cemetery_Name"]
cemetery_city_list=cemetery_info_df["city"]
cemetery_county_list=cemetery_info_df["county"]
cemetery_state_list=cemetery_info_df["state"]
cemetery_county_id_list=cemetery_info_df["county_id"]
cemetery_state_id_list=cemetery_info_df["state_id"]

field_list=['lastName','firstName','birthYear','deathYear','cemeteryName','cemeteryId','cityName','stateName']
field_list_set=set(field_list)
if append==1:print('**** Appending ****')    
if append!=1:graves_df=pd.DataFrame([])  

sleep_range=[5,12]
sleep_time_pause=1*60
pause_count=100
get_counter=0

date_df=pd.DataFrame([])

cemetery_count=0

for cemetery_id in cemetery_id_list:
    url0 = "https://www.findagrave.com/cemetery/"
    url1 ="/memorial-search?ajax=true&skip="
    url2 ="&limit=100&firstname=&middlename=&lastname=&birthyear=&birthyearfilter=&deathyear="
    url3a ="&deathyearfilter="
    url3b="&locationId=&location=&memorialid=&datefilter=&orderby=d&photofilter=&gpsfilter=&famous=&cenotaph=&monument=&isVeteran=&flowers=&sponsored=&noCemetery=&includeNickName=&includeMaidenName=&includeTitles=&hasPlot=&plot=&bio=&partialLastName=&exactName=true&linkedToName=&fuzzyNames=&mcid="

    cemetery_name=cemetery_name_list.iloc[cemetery_count]
    cemetery_city=cemetery_city_list.iloc[cemetery_count]
    cemetery_county=cemetery_county_list.iloc[cemetery_count]
    cemetery_state=cemetery_state_list.iloc[cemetery_count]
    cemetery_county_id=cemetery_county_id_list.iloc[cemetery_count]
    cemetery_state_id=cemetery_state_id_list.iloc[cemetery_count]

    cemetery_count+=1

    first_search_year=int(search_year-interval)
    last_search_year=int(search_year+interval)
    while search_year<last_year:
        frame_count=0
        grave_frame_count=0
        grave_count=100
        print(f"\n ****** {process_name:<20}• Cemetery: {cemetery_name}•{cemetery_city}•{cemetery_state} • Next Search Year: {first_search_year}-{search_year}-{last_search_year} ******* ")
        get_counter+=1
        while grave_frame_count<grave_count and interval!=0:
            url=url0+str(cemetery_id)+url1+str(grave_frame_count)+url2+str(search_year)+url3a+str(interval)+url3b
            capture_json,grave_count=get_grave_count(url)
            if status_code!=200:print('\tstatus_code:',status_code);break 
            # print('Point Frame:search_year,grave_frame_count,grave_count,interval:',search_year,grave_frame_count,grave_count,interval)
            while grave_count>10000 and interval>1:
                interval=int(interval/2)
                search_year-=interval
                first_search_year=int(search_year-interval)
                last_search_year=int(search_year+interval)
                url=url0+str(cemetery_id)+url1+str(grave_frame_count)+url2+str(search_year)+url3a+str(interval)+url3b
                capture_json,grave_count=get_grave_count(url)  
              
            grave_frame_df=extract_grave_data(capture_json) 
            if len(grave_frame_df)==0:
                if (search_year+3*interval)<=last_year:
                    search_year+=2*interval
                else:
                    interval=int((last_year-last_search_year)/2)
                    search_year=last_search_year+interval
                first_search_year=int(search_year-interval)
                last_search_year=int(search_year+interval)
                continue
            grave_frame_df=clean_dataframe(grave_frame_df)
            graves_df = graves_df._append(grave_frame_df, ignore_index = True) 
            grave_frame_count+=len(grave_frame_df)
            cemetery_grave_count+=len(grave_frame_df)

            graves_df.to_csv(fileOutInt,mode="w", index=False,header=True)
            print(f"    --- {process_name:<20}• Cemetery: {cemetery_name}•{cemetery_city}•{cemetery_state} • Search Year: {first_search_year}-{search_year}-{last_search_year} • grave count: {grave_frame_count}/{grave_count} ")
            get_counter+=1
            sleep_time=random_sleep(get_counter)       
            time.sleep(sleep_time) 
        if (search_year+3*interval)<last_year:
            search_year+=2*interval
        else:
            interval=int((last_year-last_search_year)/2)
            search_year=last_search_year+interval
        first_search_year=int(search_year-interval)
        last_search_year=int(search_year+interval)
        if interval==0:break
    print(f"***{process_name:<20}• Cemetery: {cemetery_name}•{cemetery_city}•{cemetery_state} • cemetery_id: {cemetery_id} • Cemetery Grave Count: {cemetery_grave_count} • Cemetery count: {cemetery_count}/{total_cemetery_count} \n")

        
end_time=datetime.datetime.now()
processing_time=end_time-start_time
processing_time_seconds=int(processing_time.total_seconds())
print(f"**** {process_name}   Processing Time: {processing_time_seconds} Sec")   
graves_df=clean_dataframe(graves_df)  
graves_df.to_csv(fileOut,mode="w", index=False,header=True)
graves_df.to_csv(fileOut0,mode="w", index=False,header=True) 
print("\n")
print('Uploaded To: ',fileOut0)


**Starting ***  Search_All_Jewish_Cemeteries   *** , Time: 2025-05-17 21:37:43.158277  **


 ****** Search_All_Jewish_Cemeteries • Cemetery: Hebrew Mutual Burial Ground•Philadelphia•Pennsylvania • Next Search Year: 1890-1910-1930 ******* 
    --- Search_All_Jewish_Cemeteries • Cemetery: Hebrew Mutual Burial Ground•Philadelphia•Pennsylvania • Search Year: 1890-1910-1930 • grave count: 99/99 

 ****** Search_All_Jewish_Cemeteries • Cemetery: Hebrew Mutual Burial Ground•Philadelphia•Pennsylvania • Next Search Year: 1930-1940-1950 ******* 
    --- Search_All_Jewish_Cemeteries • Cemetery: Hebrew Mutual Burial Ground•Philadelphia•Pennsylvania • Search Year: 1930-1940-1950 • grave count: 82/82 
***Search_All_Jewish_Cemeteries • Cemetery: Hebrew Mutual Burial Ground•Philadelphia•Pennsylvania • cemetery_id: 183899 • Cemetery Grave Count: 181 • Cemetery count: 1/273 


 ****** Search_All_Jewish_Cemeteries • Cemetery: Mikveh Israel Beth El Emeth Cemetery•Philadelphia•Pennsylvania • Next Search Ye

***Search_All_Jewish_Cemeteries • Cemetery: Waldheim Cemetery Co.•Forest Park•Illinois • cemetery_id: 108709 • Cemetery Grave Count: 69210 • Cemetery count: 2/1394 



In [ ]:
interval